# ResilioChain — Exploratory Data Analysis (Session 2)

## Section 1: Data Loading and Inspection

This notebook performs a full EDA on two datasets that power ResilioChain:
- inventory.csv  — daily stock records per product
- suppliers.csv  — static reference table for supplier attributes

Goal: Understand the shape, quality, types, and relationships in the data
before any modelling or feature engineering is done.

### Imports

We import the two core data science libraries used throughout this notebook.

In [2]:
import numpy as np
import pandas as pd

### Load the Data

In [3]:
inv = pd.read_csv('../../data/inventory.csv')
sup = pd.read_csv('../../data/suppliers.csv')

inv, sup         — variable names. Short and clear.
                   inv = inventory, sup = suppliers.

### Q1 — How many rows and columns does inventory.csv have?
        What does one single row represent?

PROBLEM:

Before touching any column, we need to know the size of our dataset.
Shape tells us how much data we are working with and sets expectations
for everything that follows.

In [4]:
print("Rows, Columns:", inv.shape)

Rows, Columns: (1825, 7)


Output: (1825, 7)

- 1825 rows, 7 columns
- One row = one product on one specific date
- This is a daily time-series table. Think of it as a diary:
  one entry written per product, every single day, for a full year.
- 5 products x 365 days = 1825. The maths confirms the structure.

We check suppliers.csv shape separately. It is a different kind of table.

In [5]:
print("Rows, Columns:", sup.shape)

Rows, Columns: (3, 4)


Output: (3, 4)

- 3 rows, 4 columns
- One row = one supplier and their fixed attributes
- This is NOT a time-series. It is a static reference table.
  It never changes day to day — it just holds facts about each supplier.

In database design:
  inventory.csv  = FACT table   (things that happen over time)
  suppliers.csv  = DIMENSION table (reference data, mostly static)

### Q2 — List every column and what it is measuring or tracking

PROBLEM:

Column names in raw data are often short codes. Before any analysis,
every team member must agree on what each column actually means.
This cell documents that shared understanding.

In [6]:
print("inventory.csv columns:")
print(inv.columns.tolist())

print("\nsuppliers.csv columns:")
print(sup.columns.tolist())

inventory.csv columns:
['date', 'product_id', 'closing_stock', 'stockout_flag', 'reorder_triggered', 'supplier_id', 'lead_time_days']

suppliers.csv columns:
['supplier_id', 'name', 'lead_time_days', 'reliability']


*inventory.csv* columns:
['date', 'product_id', 'closing_stock', 'stockout_flag', 
 'reorder_triggered', 'supplier_id', 'lead_time_days']

| Column            | What it tracks                                           |
|-------------------|----------------------------------------------------------|
| date              | The calendar date of that daily record                   |
| product_id        | Which product this row belongs to (P001 to P005)         |
| closing_stock     | Units left in the warehouse at the END of that day       |
| stockout_flag     | 1 = warehouse ran out of stock that day, 0 = had stock   |
| reorder_triggered | Whether a reorder order was placed (1 = yes, 0 = no)     |
| supplier_id       | Which supplier is responsible for restocking this product|
| lead_time_days    | Days between placing an order and receiving the delivery |

*suppliers.csv* columns:
['supplier_id', 'name', 'lead_time_days', 'reliability']

| Column         | What it tracks                                              |
|----------------|-------------------------------------------------------------|
| supplier_id    | Unique ID for the supplier — joins to inventory.csv         |
| name           | Full business name of the supplier                          |
| lead_time_days | Days between order and delivery for this supplier           |
| reliability    | Probability (0 to 1) that a delivery arrives on time        |

### Q3 — What is the data type of each column? What needs fixing?

PROBLEM:

When pandas reads a CSV, it guesses the type of each column.
It often gets it wrong — especially for dates, which it reads as strings.
Wrong types cause silent errors later: you cannot sort a date that is
stored as text, and you cannot do time-series operations on a string.
We must catch this now, before any analysis.

In [7]:
print("inventory.csv dtypes:")
print(inv.dtypes)

print("\nsuppliers.csv dtypes:")
print(sup.dtypes)

inventory.csv dtypes:
date                 object
product_id           object
closing_stock         int64
stockout_flag         int64
reorder_triggered     int64
supplier_id          object
lead_time_days        int64
dtype: object

suppliers.csv dtypes:
supplier_id        object
name               object
lead_time_days      int64
reliability       float64
dtype: object


*inventory.csv*:

  | Column            | Type  | Checking
  |-------------------|-------|--------------------------------------------------------|
  |date               |object |   <- WRONG. This is a date stored as a string.         |
  |product_id         |object |   <- Correct. It is a label/ID.                        | 
  |closing_stock      |int64  |   <- Correct. Count of units.                          |             
  |stockout_flag      |int64  |   <- Correct. Binary 0 or 1.                           |
  |reorder_triggered  |int64  |   <- Correct type, but column is useless (see Q16).    |
  |supplier_id        |object |   <- Correct. It is a label/ID.                        |
  |lead_time_days     |int64  |   <- Correct type, but column is redundant (see Q17).  |
                                                                                    

*suppliers.csv*:

 | Column            | Type    | Checking
 |-------------------|---------|--------------------------------------------------|
 | supplier_id       | object  |  <- Correct.                                     |
 | name              | object  |  <- Correct.                                     |
 | lead_time_days    | int64   |  <- Correct.                                     |
 | reliability       | float64 |  <- Correct. Probability values like 0.92, 0.97. |

One fix required in the inventory  table: convert 'date' from string to datetime.

In [8]:
inv['date'] = pd.to_datetime(inv['date'], dayfirst=True)
print(inv['date'].dtype)
print(inv['date'].head())

datetime64[ns]
0   2024-01-01
1   2024-01-02
2   2024-01-03
3   2024-01-04
4   2024-01-05
Name: date, dtype: datetime64[ns]


### Q4 — Are there any null values or duplicate rows?

PROBLEM:

Null values (also called NaN — Not a Number) mean missing data.
If a column has nulls, calculations on it will either fail or silently 
return wrong results. Duplicate rows mean the same event was recorded twice,
which inflates counts, sums, and averages.
Both must be checked before any analysis is trusted.

In [9]:
print("Null values in inventory.csv:")
print(inv.isnull().sum())

Null values in inventory.csv:
date                 0
product_id           0
closing_stock        0
stockout_flag        0
reorder_triggered    0
supplier_id          0
lead_time_days       0
dtype: int64


In [10]:
print("\nNull values in suppliers.csv:")
print(sup.isnull().sum())


Null values in suppliers.csv:
supplier_id       0
name              0
lead_time_days    0
reliability       0
dtype: int64


In [11]:
print("\nDuplicate rows in inventory.csv:", inv.duplicated().sum())
print("Duplicate rows in suppliers.csv:", sup.duplicated().sum())


Duplicate rows in inventory.csv: 0
Duplicate rows in suppliers.csv: 0


**Results:**

  inventory.csv  — 0 nulls in every column, 0 duplicate rows
  
  suppliers.csv  — 0 nulls in every column, 0 duplicate rows

**Important note for any analyst:**

  "No nulls and no duplicates" does NOT mean the data is ready.
  It means it has no missing or repeated records.
  Structural problems — wrong types, zero-variance columns, 
  logical inconsistencies — are completely invisible to this check.
  We address those in the sections below.

### Extra Check — Hidden whitespace in ID columns

PROBLEM:

A very common real-world data quality issue is invisible whitespace.
The value ' P001' (with a leading space) and 'P001' look identical 
in a spreadsheet but are treated as different values by Python.
This causes silent join failures and wrong group-by counts.
We must check all string ID columns now.

In [12]:
print("Whitespace in product_id:", 
      inv['product_id'].apply(lambda x: x != x.strip()).sum())

print("Whitespace in supplier_id (inventory):", 
      inv['supplier_id'].apply(lambda x: x != x.strip()).sum())

print("Whitespace in supplier_id (suppliers):", 
      sup['supplier_id'].apply(lambda x: x != x.strip()).sum())

Whitespace in product_id: 0
Whitespace in supplier_id (inventory): 0
Whitespace in supplier_id (suppliers): 0


**Result:** 0 whitespace issues across all ID columns in both files.
IDs are clean and will join correctly between the two tables.

### Extra Check — Referential Integrity between the two tables

PROBLEM:

inventory.csv has a supplier_id column that is meant to reference 
suppliers.csv. If any supplier_id in inventory does not exist in the 
suppliers table, a JOIN will silently drop those rows or produce NaN values.
This is called a referential integrity violation and it is one of the 
most common causes of wrong analytical results.
We must verify that every ID in one table exists in the other.

In [13]:
inv_suppliers = set(inv['supplier_id'].unique())
sup_suppliers = set(sup['supplier_id'].unique())

print("In inventory but NOT in suppliers:", inv_suppliers - sup_suppliers)
print("In suppliers but NOT in inventory:", sup_suppliers - inv_suppliers)

In inventory but NOT in suppliers: set()
In suppliers but NOT in inventory: set()


**Result**:
Both differences return empty sets.
Every supplier_id in inventory.csv has a matching row in suppliers.csv.
The two tables are safe to JOIN.

### Q5 — What date range does the data cover? Full year or partial?

PROBLEM:

The date range of a time-series dataset defines what patterns your model
can and cannot learn. A model trained on partial-year data will have 
seasonal blind spots — months or quarters it has never seen.

In [14]:
print("Earliest date:", inv['date'].min())
print("Latest date:  ", inv['date'].max())
print("Total unique dates:", inv['date'].nunique())

date_range = pd.date_range(start=inv['date'].min(), 
                            end=inv['date'].max(), 
                            freq='D')
missing_dates = set(date_range) - set(inv['date'].unique())
print("Missing dates in range:", len(missing_dates))

Earliest date: 2024-01-01 00:00:00
Latest date:   2024-12-30 00:00:00
Total unique dates: 365
Missing dates in range: 0


Range: 2024-01-01 to 2024-12-30
365 unique dates, 0 missing dates within that range.

This is effectively a full year of data. Only Dec 31 is absent.

Implication for forecasting:
  The model has seen almost every day of 2024.
  However, it has only seen one full cycle of seasonal patterns.
  One year is usually the minimum for seasonal modelling.
  Any predictions made for 2025 carry risk — the model has never
  seen how this supply chain behaves in a second year.

### Q6 — How many unique products? Does every product appear on every date?

PROBLEM:

In time-series data, missing rows for a product on certain dates is a 
critical problem. It can look like no activity happened, when actually 
the data was just not recorded. We need to confirm every product appears
every single day — this is called a "balanced panel".

In [15]:
print("Unique products:", inv['product_id'].nunique())
print()
print("Row count per product:")
print(inv['product_id'].value_counts())

Unique products: 5

Row count per product:
product_id
P001    365
P002    365
P003    365
P004    365
P005    365
Name: count, dtype: int64


In [16]:
print("\nProduct-to-Supplier mapping:")
print(inv.groupby('product_id')['supplier_id'].unique())


Product-to-Supplier mapping:
product_id
P001    [S001]
P002    [S002]
P003    [S003]
P004    [S001]
P005    [S002]
Name: supplier_id, dtype: object


5 unique products: P001, P002, P003, P004, P005
Each product has exactly 365 rows — perfectly balanced panel.
No missing dates for any product.

Product-to-Supplier mapping:

 |              |                                       |
 |--------------|---------------------------------------|
 | P001 -> S001 | (AsiaTech Imports,  14-day lead time) |
 | P002 -> S002 | (EuroGoods Ltd,      7-day lead time) |
 | P003 -> S003 | (LocalFast Supply,   3-day lead time) |
 | P004 -> S001 | (AsiaTech Imports,  14-day lead time) |
 | P005 -> S002 | (EuroGoods Ltd,      7-day lead time) |

Each product has a single, permanent supplier.
No dynamic supplier switching exists in this dataset.

### Q7 — closing_stock: distribution, spread, and what zero means

PROBLEM:

closing_stock is the most important numerical column in this dataset.
It directly measures warehouse health each day. Before modelling,
we need to understand its range, spread, and whether any extreme 
values indicate real operational problems or data errors.

In [17]:
print(inv['closing_stock'].describe())

count    1825.000000
mean      212.640000
std       143.644802
min         0.000000
25%        93.000000
50%       211.000000
75%       327.000000
max       595.000000
Name: closing_stock, dtype: float64


In [18]:
print("Negative stock values:", (inv['closing_stock'] < 0).sum())

Negative stock values: 0


- count    1825
- mean      212.64
- std       143.64
- min         0
- 25%        93
- 50%       211
- 75%       327
- max       595

Negative values: 0  (no data errors)

**Key observations**:
  - std (143) is almost as large as mean (212). 
    This signals high day-to-day volatility in stock levels.
    This is a hard supply chain to manage.
  - min = 0 means the warehouse physically ran empty on certain days.
    This is not a data error — it is a real stockout event.
    Every 0 represents a day where customer demand could not be met.
  - max = 595, mean = 212: the warehouse sometimes holds nearly 3x 
    the average stock level. High buffer stock is expensive to hold.

### Q8 — How often do stockouts happen? Which products are worst affected?

PROBLEM:

stockout_flag is the target variable for ResilioChain's prediction model.
Understanding how frequently stockouts occur, and which products are 
most affected, tells us both the business severity of the problem and 
whether the classes are balanced for modelling purposes.

In [19]:
print("Stockout flag distribution:")
print(inv['stockout_flag'].value_counts())
print()
print("Stockout rate: {:.2f}%".format(
    inv['stockout_flag'].mean() * 100))

Stockout flag distribution:
stockout_flag
0    1614
1     211
Name: count, dtype: int64

Stockout rate: 11.56%


In [20]:
print("Stockouts per product:")
print(inv[inv['stockout_flag'] == 1]['product_id'].value_counts())

Stockouts per product:
product_id
P001    125
P004     76
P002     10
Name: count, dtype: int64


**stockout_flag**:
  - 0 = no stockout : 1614 rows (88.4%)
  - 1 = stockout    :  211 rows (11.6%)

**Stockouts per product**:
  - P001 : 125 days  (34.2% of its own 365 days)
  - P004 :  76 days  (20.8% of its own 365 days)
  - P002 :  10 days  ( 2.7% of its own 365 days)
  - P003 :   0 days
  - P005 :   0 days

P001 and P004 are both supplied by S001 (AsiaTech Imports, 14-day lead).
The slowest supplier creates the worst stockout problem.
This is the core business problem ResilioChain is built to solve.

Modelling note: 88% vs 12% is a class imbalance.
When we build a classifier, we must handle this or the model
will learn to always predict "no stockout" and still be 88% accurate
while being completely useless.

### Extra Check — Are closing_stock and stockout_flag logically consistent?

PROBLEM:

stockout_flag should be 1 if and only if closing_stock equals 0.
If we find rows where stock is 0 but flag is 0, or stock is above 0 
but flag is 1, the data has a logical inconsistency. This kind of error
is completely invisible to null checks or duplicate checks — only a 
cross-column validation will catch it.

In [21]:
inconsistency_1 = ((inv['closing_stock'] == 0) & (inv['stockout_flag'] == 0)).sum()
inconsistency_2 = ((inv['closing_stock'] > 0)  & (inv['stockout_flag'] == 1)).sum()

print("Stock=0 but flag=0 (missed stockout):", inconsistency_1)
print("Stock>0 but flag=1 (false stockout): ", inconsistency_2)

Stock=0 but flag=0 (missed stockout): 0
Stock>0 but flag=1 (false stockout):  0


- Stock=0 but flag=0: 0
- Stock>0 but flag=1: 0

Perfect logical consistency. 
    
Every stockout flag of 1 corresponds to exactly 0 units in stock.
No missed stockouts, no false alarms in the data.
The two columns agree with each other 100%.

### Q9 — reorder_triggered: unique values and what zero variance means

PROBLEM:

A column that never changes contains no information.
In statistics this is called zero variance.
In machine learning, zero-variance features are not just useless —
they are actively harmful because a model may assign them weight,
learning noise instead of signal.

In [22]:
print("Unique values in reorder_triggered:", inv['reorder_triggered'].nunique())
print()
print(inv['reorder_triggered'].value_counts())

Unique values in reorder_triggered: 1

reorder_triggered
0    1825
Name: count, dtype: int64


Unique values: 1

Value: 0 — appears 1825 times (100% of rows)

This column never changes. It is always 0.
A column with one unique value has zero variance by definition.
It tells us absolutely nothing about any individual day or product.

What this likely means operationally:
  Either the reorder system was never triggered during this period,
  or this field was not properly recorded. Either way, it is 
  broken data and keeping it in analysis is dangerous.

### Q10 — How many suppliers? How are rows distributed across them?

In [23]:
print("Unique suppliers in inventory:", inv['supplier_id'].nunique())
print()
print("Rows per supplier:")
print(inv['supplier_id'].value_counts())

Unique suppliers in inventory: 3

Rows per supplier:
supplier_id
S001    730
S002    730
S003    365
Name: count, dtype: int64


3 unique suppliers: S001, S002, S003

 -  S001 : 730 rows  (handles P001 and P004 — 2 products x 365 days)
 -  S002 : 730 rows  (handles P002 and P005 — 2 products x 365 days)
 -  S003 : 365 rows  (handles P003 only     — 1 product  x 365 days)

Distribution is driven entirely by how many products each supplier owns.
S001 and S002 each handle 2 products, S003 handles 1.
This is not a random or imbalanced dataset — it reflects the supply chain structure.

### Q11 — lead_time_days: does it vary per row or is it fixed per product?

PROBLEM:

If lead_time_days changes day by day, it carries real signal and must be kept.
If it is always the same value for a given product, it is just a property 
of the supplier — and that property already lives in suppliers.csv.
Keeping redundant information across two tables is called data duplication
and it is a maintenance risk: if lead time changes, you now have to update 
two places instead of one.

In [24]:
print("Unique lead_time_days values:", inv['lead_time_days'].unique())
print()
print("Unique lead times per product (should be 1 if fixed):")
print(inv.groupby('product_id')['lead_time_days'].nunique())

Unique lead_time_days values: [14  7  3]

Unique lead times per product (should be 1 if fixed):
product_id
P001    1
P002    1
P003    1
P004    1
P005    1
Name: lead_time_days, dtype: int64


Unique values overall: [14, 7, 3]

Unique values per product: 1 for every single product.

lead_time_days never changes for any product.
Each product has one fixed lead time — determined entirely by its supplier.
This same information is already stored in suppliers.csv.

This column is pure redundancy. Drop candidate (confirmed in Q17).

### Q12 — suppliers.csv: rows, structure, and role in the schema

In [25]:
print(sup.shape)
print()
print(sup)

(3, 4)

  supplier_id                 name  lead_time_days  reliability
0        S001     AsiaTech Imports              14         0.92
1        S002        EuroGoods Ltd               7         0.97
2        S003  LocalFast Supply Co               3         0.99


**Shape**: (3, 4) — 3 rows, 4 columns

This is a dimension table (also called a lookup table or reference table).
It does not record events over time.
It stores fixed properties of each supplier.

You JOIN it to inventory.csv when you need supplier context:

  inv.merge(sup, on='supplier_id')
This attaches supplier name, reliability, and lead time to each 
inventory row — without duplicating data in the base table.

### Q13 — Supplier names and lead times ranked fastest to slowest

In [26]:
print(sup[['name', 'lead_time_days']].sort_values('lead_time_days'))

                  name  lead_time_days
2  LocalFast Supply Co               3
1        EuroGoods Ltd               7
0     AsiaTech Imports              14


Ranked fastest to slowest:
  1. LocalFast Supply Co  — 3 days
  2. EuroGoods Ltd        — 7 days
  3. AsiaTech Imports     — 14 days

A 14-day lead time means: if a product runs out today,
you are waiting two full weeks before new stock arrives.
This directly explains why P001 and P004 (both on AsiaTech) 
have the highest stockout counts in the dataset.

### Q14 — What does reliability measure and who is least reliable?

In [27]:
print(sup[['name', 'lead_time_days', 'reliability']].sort_values('reliability'))
print()
print("Reliability range:", sup['reliability'].min(), "to", sup['reliability'].max())
print("Any values outside 0-1:", ((sup['reliability'] > 1) | (sup['reliability'] < 0)).sum())

                  name  lead_time_days  reliability
0     AsiaTech Imports              14         0.92
1        EuroGoods Ltd               7         0.97
2  LocalFast Supply Co               3         0.99

Reliability range: 0.92 to 0.99
Any values outside 0-1: 0


All reliability values are valid (between 0 and 1).

Ranked least to most reliable:
  - AsiaTech Imports    : 0.92  (fails 8% of the time)
  - EuroGoods Ltd       : 0.97  (fails 3% of the time)
  - LocalFast Supply Co : 0.99  (fails 1% of the time)

The gap between 0.92 and 0.99 seems small.

At scale it is not:
  - AsiaTech  fails 8x more often than LocalFast.
  - Over 100 orders, AsiaTech delivers late 8 times. LocalFast: 1 time.

### Q15 — Is there a pattern between lead time and reliability?

In [28]:
print(sup[['name', 'lead_time_days', 'reliability']].sort_values('lead_time_days'))

                  name  lead_time_days  reliability
2  LocalFast Supply Co               3         0.99
1        EuroGoods Ltd               7         0.97
0     AsiaTech Imports              14         0.92


**Pattern (sorted by speed)**:
  - LocalFast Supply Co :  3 days lead time, 0.99 reliability  <- fast AND reliable
  - EuroGoods Ltd       :  7 days lead time, 0.97 reliability
  - AsiaTech Imports    : 14 days lead time, 0.92 reliability  <- slow AND unreliable

**Clear pattern**: faster suppliers are also more reliable.

**hy this happens**:
 -  Local/regional suppliers have shorter, simpler supply chains.
 -  Fewer steps = fewer points of failure = higher reliability.
 -  Long-distance suppliers (AsiaTech likely overseas) face customs,
 -  port congestion, and longer transit routes — all increase failure risk.

**This is a key insight for ResilioChain**:
  Recommending the faster supplier is simultaneously recommending
  the safer supplier. Speed and reliability are not a trade-off here — 
  they move together. LocalFast Supply Co dominates on both dimensions.

### Q16 — Which column is zero-variance and why is keeping it harmful?

PROBLEM:

Zero variance = one unique value across all rows.
A column that never changes cannot explain why one day differs from another.
It is useless for analysis and dangerous for modelling.

In [29]:
for col in inv.columns:
    print(f"{col}: {inv[col].nunique()} unique values")

date: 365 unique values
product_id: 5 unique values
closing_stock: 489 unique values
stockout_flag: 2 unique values
reorder_triggered: 1 unique values
supplier_id: 3 unique values
lead_time_days: 3 unique values


**reorder_triggered**: 1 unique value — ZERO VARIANCE.

**Why harmful, not just useless**:
1. A machine learning model may assign it a weight, learning pure noise.
2. It implies the reorder system never fired in an entire year.
   That is either a broken process or a recording failure.
   Keeping it buried in the data hides this operational finding.
3. Every analyst who opens this file will waste time asking what it means.
   Dead columns create confusion and slow down teams.

### Q17 & Q18 — Columns to drop and the final lean structure

We formally drop the two identified columns and confirm the final
analysis-ready structure of inventory.csv.

In [30]:
cols_to_drop = ['reorder_triggered', 'lead_time_days']

inv_clean = inv.drop(columns=cols_to_drop)

print("Dropped columns:", cols_to_drop)
print("New shape:", inv_clean.shape)
print("Remaining columns:", inv_clean.columns.tolist())

Dropped columns: ['reorder_triggered', 'lead_time_days']
New shape: (1825, 5)
Remaining columns: ['date', 'product_id', 'closing_stock', 'stockout_flag', 'supplier_id']


In [31]:
print(inv_clean.head())

inv_clean.to_csv('../../data/inventory.csv', index=False)

        date product_id  closing_stock  stockout_flag supplier_id
0 2024-01-01       P001            585              0        S001
1 2024-01-02       P001            571              0        S001
2 2024-01-03       P001            557              0        S001
3 2024-01-04       P001            545              0        S001
4 2024-01-05       P001            526              0        S001


**Dropped**:
 -  reorder_triggered  — zero variance. Only value is 0 across all 1825 rows.
                       Contains no information whatsoever.

 -  lead_time_days     — fully redundant. It is a fixed property of the supplier.
                       Exact same values already exist in suppliers.csv.
                       Keeping it means maintaining two sources of truth.

**Final lean structure of inventory.csv (5 columns)**:

  | Column        | Role                                          |
  |---------------|-----------------------------------------------|
  | date          | Time dimension — when the record was taken    |
  | product_id    | Entity dimension — which product              |
  | closing_stock | Core metric — warehouse health each day       |
  | stockout_flag | Target variable — what the model must predict |
  | supplier_id   | Join key — links to suppliers.csv             |

Every column that remains carries unique information 
that cannot be found anywhere else in the dataset.
This is the structure ResilioChain's model will be built on.